In [1]:
import sys
!{sys.executable} -m pip install -U google-genai python-dotenv ipywidgets


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
import os
load_dotenv() 
gemini_key = os.getenv("GEMINI_API_KEY")
print(gemini_key[:6])


AIzaSy


In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

LABEL_W = "80px"
FIELD_W = "700px"

common_style = {"description_width": LABEL_W}
text_layout = widgets.Layout(width=FIELD_W)
area_layout = widgets.Layout(width=FIELD_W, height="160px")

topic_w = widgets.Text(description="보고서 제목", layout=widgets.Layout(width="300px"))
purpose_w   = widgets.Text(description="보고서 목적", layout=widgets.Layout(width="600px"))
requirements_w = widgets.Textarea(description="요구사항", layout=widgets.Layout(width="600px", height="100px"), placeholder="예: 1. 2.")

btn = widgets.Button(description="확인", button_style="primary")
btn_box = widgets.HBox([btn], layout=widgets.Layout(justify_content="center", width="700px"))
out = widgets.Output()

result = {}  # 입력값 저장용

def on_click(_):
    result["topic"] = topic_w.value
    result["purpose"] = purpose_w.value
    result["requirements"] = requirements_w.value
    with out:
        clear_output()
        print("입력 완료")
        print(result)

btn.on_click(on_click)

display(topic_w, purpose_w, requirements_w, btn_box, out)


Text(value='', description='보고서 제목', layout=Layout(width='300px'))

Text(value='', description='보고서 목적', layout=Layout(width='600px'))

Textarea(value='', description='요구사항', layout=Layout(height='100px', width='600px'), placeholder='예: 1. 2.')

Output()

In [ ]:
import os
from google import genai
import ipywidgets as widgets
from IPython.display import display, clear_output, Markdown

# 상태 저장
toc_state = {"toc": "", "final_toc": ""}

def get_inputs(default_topic="미정(보고서제목)", default_purpose="미정(보고서목적)", default_requirements="없음"):
    r = globals().get("result", {}) or {}

    def pick(key, widget_name):
        w = globals().get(widget_name, None)
        return (r.get(key) or (getattr(w, "value", "") if w else "") or "").strip()

    topic = pick("topic", "topic_w") or default_topic
    purpose = pick("purpose", "purpose_w") or default_purpose
    requirements = pick("requirements", "requirements_w") or default_requirements
    return topic, purpose, requirements

def make_prompt(mode, topic, purpose, requirements, toc=None, feedback=None):
    """mode: 'toc' | 'revise'"""
    if mode == "toc":
        return f"""
너는 컨설팅 보고서 작성 전문가다.
아래 입력을 바탕으로 '보고서 목차(TOC)'를 한국어로 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[출력 요구사항]
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 최소 5개 장(Chapter) 이상
- 장(1,2,3...)과 소절(1.1, 1.2...)로 구성
- 불필요한 설명문 없이 목차만 출력
- #이나 * 기호 절대 사용금지
""".strip()

    if mode == "revise":
        return f"""
너는 컨설팅 보고서 편집자다.
아래 기존 목차와 사용자 수정 지시를 반영하여 '개정 목차'를 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 조건/요구사항: {requirements}

[기존 목차]
{toc}

[목차 수정]
{feedback}

[출력 요구사항]
- 번호 체계는 1, 1.1, 1.2 형태로 작성
- 불필요한 설명문 없이 개정 목차만 출력
""".strip()

    raise ValueError("mode는 'toc' 또는 'revise'여야 합니다.")

def toc_ui(model_name="gemini-2.0-flash", width="800px"):
    api_key = os.getenv("GEMINI_API_KEY")
    if not api_key:
        raise RuntimeError("GEMINI_API_KEY 환경변수가 없습니다. (.env 로드 또는 OS 환경변수 설정 필요)")

    topic, purpose, requirements = get_inputs()
    client = genai.Client(api_key=api_key)

    # 최초 목차 생성
    resp = client.models.generate_content(
        model=model_name,
        contents=make_prompt("toc", topic, purpose, requirements)
    )
    toc_state["toc"] = (resp.text or "").strip()
    toc_state["final_toc"] = toc_state["toc"]

    out = widgets.Output()

    def render():
        with out:
            clear_output()
            display(Markdown("### 생성된 목차\n\n```text\n" + toc_state["final_toc"] + "\n```"))

    feedback_w = widgets.Textarea(
        description="목차 수정",
        placeholder="예: 2장을 '시장/정책 환경'으로 변경, 3.2에 리스크 관리 추가",
        style={"description_width": "80px"},
        layout=widgets.Layout(width=width, height="120px")
    )

    apply_btn = widgets.Button(description="반영", button_style="primary")
    btn_box = widgets.HBox([apply_btn], layout=widgets.Layout(justify_content="center", width=width))

    def on_apply(_):
        feedback = (feedback_w.value or "").strip()
        if not feedback:
            toc_state["final_toc"] = toc_state["toc"]
        else:
            r = client.models.generate_content(
                model=model_name,
                contents=make_prompt("revise", topic, purpose, requirements, toc=toc_state["toc"], feedback=feedback)
            )
            toc_state["final_toc"] = (r.text or "").strip()

        # 최종 목차를 화면에 반영
        render()

    apply_btn.on_click(on_apply)

    render()
    display(out, feedback_w, btn_box)

# 실행
toc_ui(model_name="gemini-2.0-flash")


Output()

Textarea(value='', description='목차 수정', layout=Layout(height='120px', width='800px'), placeholder="예: 2장을 '시장/…

In [ ]:
# #5번 코드
import re
from google import genai


import re
def extract_chapters(final_toc: str):
    chapters = []
    pattern = re.compile(r"^\d+(?:\.\d+)*\.?\s+\S+")
    for line in (final_toc or "").splitlines():
        s = line.strip()
        s = s.replace("*", "")
        if pattern.match(s):
            chapters.append(s)
    return chapters


def make_prompt_for_report(topic, purpose, requirements, final_toc, chapter_title, summary_context=""):
    return f"""
너는 대형 컨설팅펌 수준의 전문 보고서를 작성하는 최고 수준의 분석가이다.
아래 입력 정보를 바탕으로, 지정된 “해당 장/소제목”에 대해서만
심층적이고 장문(긴 분량)의 고품질 보고서를 작성하라.

[입력]
- 보고서 제목: {topic}
- 보고서 목적: {purpose}
- 보고서 요구사항: {requirements}
- 전체 보고서 목차: {final_toc}
- 해당 장/소제목: {chapter_title}
- 이전 내용 요약: {summary_context}

[출력 요구사항]
1. REPORT
- 주어진 목차의 “해당 장/소제목”에 정확히 대응하는 본문만 작성하라.
- 단순 설명을 지양하고, 원인·구조·배경·영향·시사점의 관점에서 심층적이고 전문적인 분석을 수행하라.
- 해당 주제에 대해 객관적 사실에 기반한 정확한 분석을 제공하라.
- 관련 데이터(통계, 조사 결과, 시장 동향)와 사례(기존 연구, 산업 사례, 국가·지역 비교)를 적극 활용하여 논리를 강화하라.
- 이전 내용 요약을 참고하여 논리적 흐름이 자연스럽게 이어지도록 작성하라.
- 검증된 사실 범위 내에서 가능한 한 많은 유의미한 정보를 포함하라.
- 내용은 보고서 목적에 직접적으로 부합하도록 구성하라.
- 장 제목과 소제목은 “1. 서론”, “1.1 정의”와 같은 형태로만 작성하며, ##, ### 등 마크다운 헤더 기호는 절대 사용하지 마라.
- 문자 * 및 강조를 위한 모든 기호는 절대 사용하지 마라.
- 모든 본문은 빈 줄 없이 작성하라.
- 문단은 필요한 경우에만 구분할 수 있으며, 문단이 바뀔 때에는 단일 줄바꿈만 허용한다.
- 문단 내부의 일반 서술 문장 사이에는 줄바꿈을 사용하지 마라.
- 나열이 필요한 경우에만 줄바꿈을 허용하며,
  각 항목은 반드시 -기호로 시작하고,
  항목 간에는 단일 줄바꿈만 사용하라.
- 연속된 줄바꿈(빈 줄)은 절대 사용하지 마라.
- 소제목 바로 아래에도 빈 줄을 두지 마라.


2. SUMMARY
- 해당 장/소제목의 핵심 내용을 정확히 3문장으로 요약하라.
- 분석 결과와 시사점이 반드시 포함되도록 하라.

3. SOURCES
- 해당 장의 본문, 하위 질문, 분석 및 답변을 작성하는 과정에서
  사실 확인, 수치 인용, 분석 틀 설정, 판단 근거로 실제 사용된 모든 외부 출처를
  MLA 형식으로 기재하라.
- 직접 인용 여부와 무관하게 분석에 활용되었으면 포함하라.
- 중복 출처는 1회만 기재하라.

[출력 형식 - 반드시 아래와 같이 출력]

[REPORT]
(해당 장/소제목 보고서 본문)
[/REPORT]

[SUMMARY]
(해당 장/소제목 핵심 요약 3문장)
[/SUMMARY]

[SOURCES]
(MLA 형식 출처 목록 또는 N/A)
[/SOURCES]


""".strip()

def parse_response_blocks(text: str):
    def between(t: str, a: str, b: str) -> str:
        if a not in t or b not in t:
            return ""
        return t.split(a, 1)[1].split(b, 1)[0].strip()

    report = between(text or "", "[REPORT]", "[/REPORT]")
    summary = between(text or "", "[SUMMARY]", "[/SUMMARY]")
    sources = between(text or "", "[SOURCES]", "[/SOURCES]")
    return report, summary, sources


topic, purpose, requirements = get_inputs()
final_toc = (toc_state.get("final_toc") or "").strip()

final_report = f"{topic}\n\n{final_toc}\n\n"
final_summary = ""
final_sources = ""
summary_context = ""

def generate_report_from_toc(final_toc, topic, purpose, requirements, api_key, summary_context, model="gemini-2.5-pro"):
    """
    - summary만(함수 내부) 초기화해서 이번 실행분 요약만 return
    - report/sources는 함수에서 생성해서 return
    - summary_context는 입력으로 받아, 이번 실행에서 추가된 내용을 반영한 updated_summary_context를 return
    """
    client = genai.Client(api_key=api_key)

    chapters = extract_chapters(final_toc)

    report = ""
    summary = ""
    sources = ""

    updated_summary_context = summary_context or ""

    for chapter_title in chapters:
        prompt = make_prompt_for_report(
            topic, purpose, requirements, final_toc, chapter_title, updated_summary_context
        )

        response = client.models.generate_content(model=model, contents=prompt)
        r, s, src = parse_response_blocks(getattr(response, "text", "") or "")

        report += f"{r}\n\n"
        sources += f"{src}\n"

        updated_summary_context += f"- {chapter_title}: {s}\n"

    return report, summary, sources, updated_summary_context


# ===== 사용 예시: summary_context는 함수 밖에서 누적 관리 =====
report, summary_content, sources, summary_context = generate_report_from_toc(
    final_toc=toc_state["final_toc"],
    topic=topic,
    purpose=purpose,
    requirements=requirements,
    api_key=gemini_key,
    summary_context=summary_context,   # ✅ 이전까지 누적된 컨텍스트를 넣고
    model="gemini-2.5-pro",
)


final_report += report
final_sources += sources


print("==== final_report (first 500 chars) ====")
print(final_report[:500])

print("\n==== final_sources (first 500 chars) ====")
print(final_sources[:500])

print("\n==== summary_context (first 500 chars) ====")
print(summary_context[:500])


==== final_report (first 500 chars) ====
미정(보고서 제목)

## 개정 목차: 미국 캘리포니아 가스 피커 발전소 전망

1. 서론
    1.1. 보고서 개요 및 목적
    1.2. 캘리포니아 전력 시장 현황
    1.3. 가스 피커 발전소의 역할 및 중요성

2. 캘리포니아 가스 피커 발전 시장 분석
    2.1. 시장 규모 및 성장 동력
    2.2. 경쟁 환경 분석
    2.3. 규제 환경 변화 및 영향
    2.4. 기술 동향 및 혁신

3. 2027년-2030년 가스 피커 발전소 매출 전망
    3.1. 수요 예측 (전력 수요, 피크 수요)
    3.2. 가격 전망 (천연가스 가격, 전력 가격)
    3.3. 매출 시나리오 분석 (고성장, 기준, 저성장)

4. 2027년-2030년 가스 피커 발전소 수익성 분석
    4.1. 비용 구조 분석 (운영 비용, 유지보수 비용, 연료 비용)
    4.2. 수익성 지표 분석 (EBITDA, 순이익)
    4.3. 수익성 개선 방안

5. 투자 및 운

==== final_sources (first 500 chars) ====
California Independent System Operator (CAISO). "The Duck Curve - What it is and why it's a challenge." *CAISO*, 2022, www.caiso.com/documents/flexibleresourceshelpalleviateovergeneration-duckcurve.pdf.
California State Legislature. *Senate Bill No. 100*. 10 Sept. 2018, leginfo.legislature.ca.gov/faces/billNavClient.xhtml?bill_id=201720180SB100.
U.S. Energy Information Administration. "California State Energy Pr

In [8]:
from docx import Document
import os
import re

def sanitize_filename(name: str, default="report"):
    name = (name or "").strip() or default
    # Windows 금지 문자 제거
    name = re.sub(r'[\\/:*?"<>|]', "_", name)
    # 너무 길면 잘라내기(옵션)
    return name[:150]

def get_download_path(file_name="report.docx"):
    if os.name == "nt":  # Windows
        download_path = os.path.join(os.environ.get("USERPROFILE", ""), "Downloads")
    else:  # macOS, Linux
        download_path = os.path.join(os.environ.get("HOME", ""), "Downloads")

    if not download_path or not os.path.isdir(download_path):
        # Downloads가 없으면 현재 폴더로 fallback
        download_path = os.getcwd()

    os.makedirs(download_path, exist_ok=True)
    return os.path.join(download_path, file_name)

def add_multiline_text(doc: Document, text: str):
    for line in (text or "").splitlines():
        if line.strip() == "":
            doc.add_paragraph("")  # 빈 줄 유지
        else:
            doc.add_paragraph(line)

def save_report_and_sources(final_report, final_sources, topic):
    file_name = f"{sanitize_filename(topic)}.docx"
    file_path = get_download_path(file_name)

    doc = Document()

    doc.add_heading("Report", level=1)
    add_multiline_text(doc, final_report)

    doc.add_heading("Sources", level=1)
    add_multiline_text(doc, final_sources)

    doc.save(file_path)
    print(f"파일이 저장되었습니다: {file_path}")

save_report_and_sources(final_report, final_sources, topic)


파일이 저장되었습니다: C:\Users\mphk0\Downloads\미정(보고서 제목).docx
